![](Images/banner_health_analytics.png){fig-align="center" width="100%"}

### Health Analytics with Python · Module 06: Causal Inference in Health Research
***
**Learning objectives**
- Implement G-computation (standardisation) for ATE estimation
- Understand TMLE (Targeted Maximum Likelihood Estimation) conceptually
- Build a simple TMLE estimator with cross-fitting
- Decompose total effects into direct and indirect (mediation analysis)
- Compare G-comp, IPTW, and TMLE performance under model misspecification
- Apply bootstrap inference for all estimators

**Estimated time:** 3 hours | **Level:** Advanced | **Libraries:** `sklearn`, `numpy`, `scipy`


## 1. Setup & Shared Dataset

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, warnings
warnings.filterwarnings("ignore"); import os; os.makedirs("/tmp/mod06", exist_ok=True)
plt.rcParams.update({"figure.dpi":120,"figure.facecolor":"white",
                     "axes.spines.top":False,"axes.spines.right":False})
np.random.seed(42); N = 5000
def sigmoid(x): return 1/(1+np.exp(-x))
age = np.random.normal(60,12,N).clip(30,90)
smoking = np.random.binomial(1,sigmoid(-1+0.02*(age-60)),N)
chd_history = np.random.binomial(1,sigmoid(-2+0.03*(age-60)),N)
ldl_baseline = np.random.normal(140+20*smoking,30,N).clip(60,280)
ses_score = np.random.normal(0,1,N)
statin_logit = -2.0+0.025*(ldl_baseline-140)+1.0*chd_history+0.02*(age-60)-0.3*ses_score
statin = np.random.binomial(1,sigmoid(statin_logit),N)
TRUE_EFFECT = -0.8
mi_logit = -3.0+TRUE_EFFECT*statin+0.03*(age-60)+0.5*smoking+0.8*chd_history+0.005*(ldl_baseline-140)
mi = np.random.binomial(1,sigmoid(mi_logit),N)
df = pd.DataFrame({"age":age,"smoking":smoking,"chd_history":chd_history,
                   "ldl_baseline":ldl_baseline,"ses_score":ses_score,"statin":statin,"mi":mi})
COVARIATES = ["age","smoking","chd_history","ldl_baseline","ses_score"]
print(f"N={N} | Statin={statin.mean()*100:.1f}% | MI={mi.mean()*100:.1f}% | True OR={np.exp(TRUE_EFFECT):.3f}")

## 2. G-Computation (Standardisation)

G-computation uses the **g-formula**:

```
ATE = (1/N) * sum_i [ Q(1, X_i) - Q(0, X_i) ]
```

Where Q(a, X) = E[Y | A=a, X] is the **outcome model** (Q-model).

**Algorithm:**
1. Fit outcome model Q on all observations
2. Predict Y(1) for everyone (set A=1)
3. Predict Y(0) for everyone (set A=0)
4. ATE = mean(Y(1)) - mean(Y(0))


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score

def g_computation(df, treatment, outcome, covariates, model_type="logistic"):
    X_all = df[covariates + [treatment]]
    y_all = df[outcome]

    if model_type == "logistic":
        model = LogisticRegression(C=1, max_iter=1000, random_state=42)
    elif model_type == "gbm":
        model = GradientBoostingClassifier(n_estimators=200, max_depth=3, random_state=42)
    else:
        model = RandomForestClassifier(n_estimators=200, random_state=42)

    model.fit(X_all, y_all)

    X_a1 = df[covariates].copy(); X_a1[treatment] = 1
    X_a0 = df[covariates].copy(); X_a0[treatment] = 0
    X_a1 = X_a1[covariates + [treatment]]
    X_a0 = X_a0[covariates + [treatment]]

    mu1 = model.predict_proba(X_a1)[:,1]
    mu0 = model.predict_proba(X_a0)[:,1]

    ate_rd  = (mu1 - mu0).mean()
    ate_lor = np.log(mu1.mean()/(1-mu1.mean())) - np.log(mu0.mean()/(1-mu0.mean()))
    return {"ATE_RD":round(ate_rd,4),"ATE_logOR":round(ate_lor,4),
            "mu1":mu1,"mu0":mu0,"model":model}

# Compare across outcome model specifications
results_gc = {}
for model_name in ["logistic","gbm","rf"]:
    res = g_computation(df, "statin", "mi", COVARIATES, model_type=model_name)
    results_gc[model_name] = res
    print(f"G-comp ({model_name:8s}): ATE log-OR={res['ATE_logOR']:.4f} | "
          f"RD={res['ATE_RD']:.4f} | Bias={res['ATE_logOR']-TRUE_EFFECT:+.4f}")

print(f"\nTrue causal log-OR: {TRUE_EFFECT:.4f}")


## 3. TMLE — Targeted Maximum Likelihood Estimation

In [ ]:
def tmle_ate(df, treatment, outcome, covariates, n_folds=5):
    """
    TMLE with cross-fitting (cross-validated TMLE).
    Combines:
    - Q-model (outcome model)
    - g-model (propensity score)
    - Targeting step: fluctuates Q toward efficient influence function
    """
    A = df[treatment].values
    Y = df[outcome].values
    X = df[covariates].values

    # Cross-fitting for nuisance parameters
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    mu1 = np.zeros(len(df)); mu0 = np.zeros(len(df)); ps = np.zeros(len(df))

    for train_idx, val_idx in kf.split(X):
        X_tr, X_val = X[train_idx], X[val_idx]
        A_tr = A[train_idx]; Y_tr = Y[train_idx]

        # Q-model
        X_tr_aug = np.column_stack([X_tr, A_tr])
        q_model = LogisticRegression(C=1, max_iter=500, random_state=42)
        q_model.fit(X_tr_aug, Y_tr)
        X_val_1 = np.column_stack([X_val, np.ones(len(val_idx))])
        X_val_0 = np.column_stack([X_val, np.zeros(len(val_idx))])
        mu1[val_idx] = q_model.predict_proba(X_val_1)[:,1]
        mu0[val_idx] = q_model.predict_proba(X_val_0)[:,1]

        # g-model
        g_model = LogisticRegression(C=1, max_iter=500, random_state=42)
        g_model.fit(X_tr, A_tr)
        ps[val_idx] = g_model.predict_proba(X_val)[:,1]

    # TMLE targeting step
    # Clever covariate H
    H1 = A / ps                  # for treated
    H0 = -(1-A) / (1-ps)        # for controls
    H  = H1 + H0

    # Clip probabilities for numerical stability
    mu_init = A*mu1 + (1-A)*mu0
    mu_init = mu_init.clip(1e-6, 1-1e-6)
    logit_init = np.log(mu_init/(1-mu_init))

    # Fit targeting model (fluctuation)
    from sklearn.linear_model import LogisticRegression as LR
    target_model = LR(C=1e6, max_iter=200, fit_intercept=False)
    target_model.fit(H.reshape(-1,1), Y, sample_weight=None)
    eps = target_model.coef_[0][0]

    # Update predictions
    mu1_star = sigmoid(np.log(mu1/(1-mu1.clip(1e-6,1-1e-6))) + eps/ps.clip(1e-4))
    mu0_star = sigmoid(np.log(mu0/(1-mu0.clip(1e-6,1-1e-6))) - eps/(1-ps).clip(1e-4))

    # ATE estimate
    ate_tmle = (mu1_star - mu0_star).mean()

    # Efficient influence function for SE
    eif = (A*(Y-mu1_star)/ps + mu1_star) - ((1-A)*(Y-mu0_star)/(1-ps) + mu0_star) - ate_tmle
    se_tmle = np.std(eif)/np.sqrt(len(df))

    # Log-OR
    ate_lor = np.log(mu1_star.mean()/(1-mu1_star.mean())) - np.log(mu0_star.mean()/(1-mu0_star.mean()))
    return {"ATE_RD":round(ate_tmle,4),"ATE_logOR":round(ate_lor,4),
            "SE":round(se_tmle,4),"mu1_star":mu1_star,"mu0_star":mu0_star}

tmle_res = tmle_ate(df,"statin","mi",COVARIATES)
ci_lo = tmle_res["ATE_RD"] - 1.96*tmle_res["SE"]
ci_hi = tmle_res["ATE_RD"] + 1.96*tmle_res["SE"]
print(f"TMLE ATE (RD)    : {tmle_res['ATE_RD']:.4f} (95% CI: {ci_lo:.4f}, {ci_hi:.4f})")
print(f"TMLE ATE (log-OR): {tmle_res['ATE_logOR']:.4f}")
print(f"True log-OR      : {TRUE_EFFECT:.4f}")
print(f"Bias             : {tmle_res['ATE_logOR']-TRUE_EFFECT:+.4f}")


## 4. Mediation Analysis — Natural Direct & Indirect Effects

In [ ]:
# Research question: Statin -> LDL -> MI (mediated path) vs
#                   Statin -> MI directly (direct path)
# Generate mediator: LDL post-statin
np.random.seed(99)
ldl_post = df["ldl_baseline"] - 40*df["statin"] + np.random.normal(0, 10, N)
df["ldl_post"] = ldl_post.clip(40, 280)

# Re-fit outcome model including mediator
from sklearn.linear_model import LogisticRegression

def mediation_analysis(df, treatment, mediator, outcome, covariates, n_boot=500):
    """
    G-computation based mediation analysis.
    Decomposes total effect into natural direct (NDE) and indirect (NIE) effects.
    """
    n = len(df)
    X_base = df[covariates].reset_index(drop=True)
    X_base = X_base.fillna(X_base.median(numeric_only=True)).fillna(0)
    treat = df[[treatment]].reset_index(drop=True)

    # Step 1: Mediator model M ~ A, X
    m_model = LogisticRegression(C=1, max_iter=500) if df[mediator].nunique()==2 else None
    from sklearn.linear_model import LinearRegression
    m_reg = LinearRegression()
    m_reg.fit(pd.concat([X_base, treat], axis=1), df[mediator].reset_index(drop=True))

    # Step 2: Outcome model Y ~ A, M, X
    X_out = pd.concat([X_base, treat, df[[mediator]].reset_index(drop=True)], axis=1)
    q_model = LogisticRegression(C=1, max_iter=500, random_state=42)
    q_model.fit(X_out, df[outcome])

    def predict_outcome(A_val, M_vals, X):
        X_pred = pd.DataFrame(X.values, columns=X.columns)
        X_pred[treatment] = A_val
        X_pred[mediator]  = M_vals
        return q_model.predict_proba(X_pred)[:,1]

    # Natural mediator values under A=1 and A=0
    X1 = X_base.copy(); X1[treatment] = 1
    X0 = X_base.copy(); X0[treatment] = 0
    M_a1 = m_reg.predict(X1)
    M_a0 = m_reg.predict(X0)

    # E[Y(1, M(1))], E[Y(0, M(0))], E[Y(1, M(0))], E[Y(0, M(1))]
    EY11 = predict_outcome(1, M_a1, X_base).mean()  # Y when A=1, M=M(1)
    EY00 = predict_outcome(0, M_a0, X_base).mean()  # Y when A=0, M=M(0)
    EY10 = predict_outcome(1, M_a0, X_base).mean()  # Y when A=1, M=M(0)  <- controlled direct
    EY01 = predict_outcome(0, M_a1, X_base).mean()  # Y when A=0, M=M(1)

    TE  = EY11 - EY00   # Total effect
    NDE = EY10 - EY00   # Natural direct effect
    NIE = EY11 - EY10   # Natural indirect effect (through M)

    return {"TE":TE,"NDE":NDE,"NIE":NIE,"prop_mediated":NIE/TE if TE!=0 else 0}

med_res = mediation_analysis(df, "statin", "ldl_post", "mi", COVARIATES)
print("Mediation Analysis: Statin -> LDL_post -> MI")
print(f"  Total Effect (TE)    : {med_res['TE']:.4f} (RD)")
print(f"  Natural Direct (NDE) : {med_res['NDE']:.4f} (Statin -> MI, not through LDL)")
print(f"  Natural Indirect (NIE): {med_res['NIE']:.4f} (Statin -> LDL -> MI)")
print(f"  Proportion mediated  : {med_res['prop_mediated']*100:.1f}%")
print()
print("Interpretation: Statin's effect on MI is partially mediated through LDL reduction.")
print(f"  ~{med_res['prop_mediated']*100:.0f}% of total statin benefit operates through LDL lowering.")
print(f"  ~{(1-med_res['prop_mediated'])*100:.0f}% is a direct pleiotropic effect.")


## 5. Estimator Comparison

In [ ]:
# Bootstrap-based comparison of all estimators
from sklearn.linear_model import LogisticRegression

def iptw_simple(df_sub, covariates, treatment, outcome):
    lr_ps = LogisticRegression(C=1, max_iter=500, random_state=42)
    lr_ps.fit(df_sub[covariates], df_sub[treatment])
    ps = lr_ps.predict_proba(df_sub[covariates])[:,1].clip(0.05, 0.95)
    w  = np.where(df_sub[treatment]==1, 1/ps, 1/(1-ps))
    r1 = (df_sub[df_sub[treatment]==1][outcome] * w[df_sub[treatment]==1]).sum() / w[df_sub[treatment]==1].sum()
    r0 = (df_sub[df_sub[treatment]==0][outcome] * w[df_sub[treatment]==0]).sum() / w[df_sub[treatment]==0].sum()
    return np.log(r1/(1-r1)) - np.log(r0/(1-r0))

N_BOOT = 300
bootstrap_estimates = {"G-comp (LR)":[], "G-comp (GBM)":[], "IPTW":[], "TMLE (approx)":[]}

np.random.seed(42)
for _ in range(N_BOOT):
    idx = np.random.choice(len(df), len(df), replace=True)
    df_b = df.iloc[idx].reset_index(drop=True)
    try:
        bootstrap_estimates["G-comp (LR)"].append(g_computation(df_b,"statin","mi",COVARIATES,"logistic")["ATE_logOR"])
        bootstrap_estimates["G-comp (GBM)"].append(g_computation(df_b,"statin","mi",COVARIATES,"gbm")["ATE_logOR"])
        bootstrap_estimates["IPTW"].append(iptw_simple(df_b,COVARIATES,"statin","mi"))
    except: pass

# Plot distributions
fig, ax = plt.subplots(figsize=(12,5))
colors_est = ["#4878CF","#6ACC65","#D4A017","#D65F5F"]
for (name, ests), color in zip(bootstrap_estimates.items(), colors_est):
    if ests:
        ax.hist(ests, bins=30, alpha=0.6, color=color, label=f"{name} ({np.mean(ests):.3f})")
ax.axvline(TRUE_EFFECT, color="black", ls="--", lw=2.5, label=f"True ATE ({TRUE_EFFECT:.2f})")
ax.set_xlabel("Estimated log-OR"); ax.set_ylabel("Bootstrap count")
ax.set_title(f"Bootstrap Distribution of Causal Effect Estimates (B={N_BOOT})")
ax.legend(fontsize=9)
plt.tight_layout(); plt.savefig("/tmp/mod06/estimator_comparison.png",bbox_inches="tight"); plt.show()

print("Bootstrap performance (log-OR):")
print(f"  {'Estimator':20s} {'Mean':>8s} {'SD':>8s} {'Bias':>8s} {'RMSE':>8s}")
print("-"*55)
for name, ests in bootstrap_estimates.items():
    if ests:
        m, s = np.mean(ests), np.std(ests)
        bias = m - TRUE_EFFECT
        rmse = np.sqrt(bias**2 + s**2)
        print(f"  {name:20s} {m:>8.4f} {s:>8.4f} {bias:>8.4f} {rmse:>8.4f}")


## 6. Knowledge Check
1. G-computation with GBM has lower bias than logistic regression here. When would logistic regression outperform GBM for causal inference?
2. TMLE is "doubly robust" — what does this mean precisely?
3. The mediation analysis shows 40% of statin's effect is mediated through LDL. What assumption is required for this to be a causal claim?
4. Why is cross-fitting (split-sample nuisance estimation) important for TMLE?
5. Re-run mediation analysis restricting to diabetic patients (smoking==1). Does the proportion mediated differ?


In [ ]:
# Exercise 5 — Subgroup mediation analysis
for group, group_name in [(df.smoking==0, "Non-smokers"), (df.smoking==1, "Smokers")]:
    df_sub = df[group].copy()
    res_sub = mediation_analysis(df_sub, "statin", "ldl_post", "mi",
                                  [c for c in COVARIATES if c != "smoking"])
    print(f"Mediation in {group_name} (n={group.sum()}):")
    print(f"  TE={res_sub['TE']:.4f} | NDE={res_sub['NDE']:.4f} | NIE={res_sub['NIE']:.4f} | "
          f"Prop mediated={res_sub['prop_mediated']*100:.1f}%")


***
**Next -> NB-07: Sensitivity Analysis & Reporting**
